# S6E11 GPU · FT-Transformer cho dữ liệu bảng

Thí nghiệm dự đoán gian lận bằng dữ liệu cuộc thi và mô hình tự huấn luyện. Mỗi phần có giải thích trước mã để có thể đọc và chạy theo thứ tự. Điểm được đo bằng năm phần chia trên `train.csv`.

Mở notebook từ bản sao repo và chọn kernel Python có thư viện trong
[Hướng dẫn chuẩn bị](README.md). Dữ liệu chính thức được đọc cục bộ và đối chiếu SHA-256.

## 1. Dữ liệu và môi trường GPU

Nguồn nhãn là `train.csv` chính thức của S6E11. Mã kiểm tra SHA-256 phải khớp
bản dữ liệu của repo. Tập kiểm tra của Kaggle không có nhãn và ROC AUC trong
notebook được đo trên các giao dịch giữ lại từ tập huấn luyện có nhãn.
`id` chỉ dùng để truy vết. Mỗi mô hình xác nhận thiết bị GPU trước khi fit.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from subprocess import check_output
import sys
from time import perf_counter
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
SMOKE = os.environ.get('S6E11_SMOKE') == '1'

def locate_repo():
    override = os.environ.get('S6E11_REPO')
    start = Path.cwd().resolve()
    roots = (Path(override).expanduser().resolve(),) if override else (start, *start.parents)
    for root in roots:
        if (root / 'configs' / 'default.json').is_file() and (root / 'data' / 'manifest.json').is_file():
            return root
    raise FileNotFoundError('Mở notebook trong repo S6E11 hoặc đặt S6E11_REPO.')


def locate_data(repo):
    override = os.environ.get('S6E11_DATA_DIR')
    roots = (Path(override).expanduser().resolve(),) if override else (repo / 'data', repo.parent)
    for root in roots:
        if any((root / name).is_file() for name in ('train.csv', 'playground-series-s6e11.zip')):
            return root
    raise FileNotFoundError('Cần train.csv hoặc ZIP S6E11 chính thức. Đặt S6E11_DATA_DIR.')


REPO = locate_repo()
DATA_DIR = locate_data(REPO)
default_output = REPO / 's6e11_gpu' / 'model_notebooks' / 'local_results'
OUTPUT_ROOT = Path(os.environ.get('S6E11_NOTEBOOK_OUTPUT', str(default_output))).expanduser().resolve()
ROUND1_DIR = OUTPUT_ROOT / 'round1'
ROUND2_DIR = OUTPUT_ROOT / 'round2'
CONFIG = json.loads((REPO / 'configs' / 'default.json').read_text(encoding='utf-8-sig'))
MANIFEST = json.loads((REPO / 'data' / 'manifest.json').read_text(encoding='utf-8-sig'))
TARGET = CONFIG['target']
NUMERIC = CONFIG['numeric_columns']
FLAGS = CONFIG['flag_columns']
CATEGORICAL = CONFIG['categorical_columns']

GPU_NAME = check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        text=True, timeout=15).splitlines()[0].strip()
if 'RTX 4060' not in GPU_NAME:
    raise RuntimeError(f'GPU hiện tại khác RTX 4060 đã kiểm tra: {GPU_NAME}')
print(f'GPU phát hiện: {GPU_NAME}')

def load_competition_train():
    expected_hash = MANIFEST['input_files']['train.csv']['sha256']
    csv_path = DATA_DIR / 'train.csv'
    if csv_path.is_file():
        with csv_path.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv không khớp manifest')
        return pd.read_csv(csv_path), digest

    with ZipFile(DATA_DIR / 'playground-series-s6e11.zip') as archive:
        with archive.open('train.csv') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        if digest != expected_hash:
            raise ValueError('SHA-256 của train.csv trong ZIP không khớp manifest')
        with archive.open('train.csv') as stream:
            return pd.read_csv(stream), digest

frame, DATA_HASH = load_competition_train()
feature_names = NUMERIC + FLAGS + CATEGORICAL
required = ['id', TARGET, *feature_names]
if (set(frame.columns) != set(required) or len(frame) != MANIFEST['train_rows']
        or frame.id.duplicated().any() or frame[required].isna().any().any()
        or set(frame[TARGET].unique()) != {0, 1}):
    raise ValueError('Dữ liệu huấn luyện không khớp schema đã kiểm tra của repo')
if SMOKE:
    _, frame = train_test_split(
        frame, test_size=20_000, stratify=frame[TARGET], random_state=SEED,
    )
    frame = frame.reset_index(drop=True)
y = frame[TARGET].to_numpy(dtype=np.uint8)
ids = frame['id'].to_numpy(dtype=np.int64)
X = frame[feature_names]
raw_numeric = X[NUMERIC + FLAGS].to_numpy(dtype=np.float32)
if not np.isfinite(raw_numeric).all():
    raise ValueError('Đặc trưng số có giá trị không hữu hạn')
print(f'Số dòng: {len(y):,} | nhãn gian lận: {y.sum():,} ({y.mean():.2%})')

## 2. Năm phần chia phân tầng

Dùng `StratifiedKFold` với seed 42 như repo để giữ tỷ lệ gian lận giữa các phần.
Mỗi giao dịch được chấm một lần bởi mô hình học từ bốn phần còn lại.
Số vòng hoặc số epoch được cố định trước khi đánh giá phần giữ lại.

In [ ]:
n_folds = 2 if SMOKE else 5
splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
splits = list(splitter.split(X, y))
assignment = np.full(len(y), -1, dtype=np.int8)
for fold, (_, val_idx) in enumerate(splits):
    assignment[val_idx] = fold
if (assignment < 0).any():
    raise RuntimeError('Có giao dịch chưa được gán fold')
SPLIT_HASH = hashlib.sha256(ids.astype('<i8').tobytes() + assignment.tobytes()).hexdigest()
print(f'{n_folds} fold | mã kiểm tra cách chia: {SPLIT_HASH[:16]}')

## 3. Chuẩn hóa đầu vào cho mạng nơ-ron

Trung bình, độ lệch chuẩn và bảng giá trị phân loại được xác định từ phần
huấn luyện. Các giá trị phân loại chưa gặp dùng mã 0. Mạng học các vectơ biểu
diễn (embeddings) của cột phân loại trực tiếp từ nhãn huấn luyện.

In [ ]:
def prepare_dl_fold(train_idx, val_idx):
    raw_train, raw_val = raw_numeric[train_idx], raw_numeric[val_idx]
    center = raw_train.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = raw_train.std(axis=0, dtype=np.float64).astype(np.float32)
    scale[scale < 1e-7] = 1.0
    num_train = (raw_train - center) / scale
    num_val = (raw_val - center) / scale
    cat_train = np.empty((len(train_idx), len(CATEGORICAL)), dtype=np.int64)
    cat_val = np.empty((len(val_idx), len(CATEGORICAL)), dtype=np.int64)
    sizes = []
    for col, name in enumerate(CATEGORICAL):
        values = X[name].to_numpy()
        vocab = pd.Index(pd.unique(values[train_idx]))
        sizes.append(len(vocab) + 1)
        cat_train[:, col] = vocab.get_indexer(values[train_idx]) + 1
        cat_val[:, col] = vocab.get_indexer(values[val_idx]) + 1
    return (num_train, cat_train), (num_val, cat_val), sizes

## 4. Vòng lặp huấn luyện CUDA

Đưa các tensor lên GPU và cập nhật trọng số theo từng nhóm giao dịch.

In [ ]:
import time

import torch
from torch import nn


class CategoricalEmbedding(nn.Module):
    def __init__(self, category_sizes, width):
        super().__init__()
        offsets = np.cumsum([0, *category_sizes], dtype=np.int64)[:-1]
        self.register_buffer("offsets", torch.from_numpy(offsets))
        self.embedding = nn.Embedding(max(1, sum(category_sizes)), width)
        nn.init.normal_(self.embedding.weight, std=0.02)

    def forward(self, categories):
        return self.embedding(categories + self.offsets)


def train_torch_fold(model_class, train_idx, val_idx, fold, *, epochs, batch_size, learning_rate):
    if not torch.cuda.is_available():
        raise RuntimeError("Notebook này cần PyTorch có CUDA và GPU NVIDIA đang hoạt động.")
    device = torch.device("cuda", torch.cuda.current_device())
    gpu_name = torch.cuda.get_device_name(device)
    if gpu_name != GPU_NAME:
        raise RuntimeError(f"Thiết bị PyTorch không khớp GPU đã kiểm tra: {gpu_name}")
    seed = SEED + int(fold)
    torch.manual_seed(seed)
    sampler = torch.Generator(device=device).manual_seed(seed)
    use_bfloat16 = torch.cuda.is_bf16_supported()

    (train_num, train_cat), (val_num, val_cat), category_sizes = prepare_dl_fold(train_idx, val_idx)
    train_num = torch.as_tensor(train_num, dtype=torch.float32, device=device)
    train_cat = torch.as_tensor(train_cat, dtype=torch.long, device=device)
    train_y = torch.as_tensor(y[train_idx], dtype=torch.float32, device=device)

    torch.cuda.synchronize(device)
    started = time.perf_counter()
    model = model_class(train_num.shape[1], category_sizes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4, fused=True)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(train_y), device=device, generator=sampler)
        for rows in order.split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
                logits = model(train_num[rows], train_cat[rows])
                loss = loss_fn(logits, train_y[rows])
            loss.backward()
            optimizer.step()

    torch.cuda.synchronize(device)
    fit_seconds = time.perf_counter() - started

    val_num = torch.as_tensor(val_num, dtype=torch.float32, device=device)
    val_cat = torch.as_tensor(val_cat, dtype=torch.long, device=device)
    scores = torch.empty(len(val_num), dtype=torch.float32, device=device)
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(val_num), batch_size):
            rows = slice(start, start + batch_size)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bfloat16):
                logits = model(val_num[rows], val_cat[rows])
            scores[rows] = logits.float().sigmoid()

    return scores.cpu().numpy(), {
        "device": f"{device}: {gpu_name} (PyTorch CUDA)",
        "fit_seconds": fit_seconds,
        "epochs": epochs,
        "batch_size": batch_size,
        "precision": "bfloat16 AMP" if use_bfloat16 else "float32",
    }

## FT-Transformer trên GPU

FT-Transformer (Feature Tokenizer Transformer) biến mỗi cột thành một
véc-tơ đại diện cho đặc trưng. Mỗi cột số có bộ trọng số và độ lệch riêng.
Các cột phân loại dùng embedding theo giá trị và độ lệch riêng theo cột.
Một khối Transformer với 4 đầu chú ý học quan hệ giữa các cột. Lớp cuối
đọc véc-tơ tổng hợp `[CLS]` để tính điểm số cho mỗi giao dịch.

Mỗi fold huấn luyện theo lịch cố định 3 epoch (3 lượt qua tập huấn luyện),
hoặc 1 epoch khi chạy thử. Mô hình tối ưu binary cross-entropy với trọng số
như nhau cho mỗi dòng và dự đoán tập đánh giá sau epoch cuối. Thời gian
`fit_seconds` đo phần khởi tạo và huấn luyện mô hình trên GPU.

In [ ]:
class FraudFTTransformer(nn.Module):
    def __init__(self, num_features, category_sizes):
        super().__init__()
        width = 32
        self.numeric_weight = nn.Parameter(torch.empty(num_features, width))
        self.numeric_bias = nn.Parameter(torch.zeros(num_features, width))
        self.categories = CategoricalEmbedding(category_sizes, width)
        self.category_bias = nn.Parameter(torch.zeros(len(category_sizes), width))
        self.cls_token = nn.Parameter(torch.empty(1, 1, width))
        nn.init.normal_(self.numeric_weight, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)
        self.attention = nn.TransformerEncoderLayer(
            d_model=width,
            nhead=4,
            dim_feedforward=64,
            dropout=0.10,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.output = nn.Sequential(nn.LayerNorm(width), nn.GELU(), nn.Linear(width, 1))

    def forward(self, numeric, categorical):
        numeric_tokens = numeric.unsqueeze(-1) * self.numeric_weight + self.numeric_bias
        category_tokens = self.categories(categorical) + self.category_bias
        cls_tokens = self.cls_token.expand(len(numeric), -1, -1)
        tokens = torch.cat((cls_tokens, numeric_tokens, category_tokens), dim=1)
        encoded = self.attention(tokens)
        return self.output(encoded[:, 0]).squeeze(-1)


def fit_predict(train_idx, val_idx, fold):
    return train_torch_fold(
        FraudFTTransformer, train_idx, val_idx, fold,
        epochs=1 if SMOKE else 3, batch_size=1024, learning_rate=6e-4,
    )

## Đánh giá

ROC AUC đo chất lượng xếp hạng giao dịch gian lận. Điểm OOF gộp dùng các dự
đoán từ cả năm phần giữ lại. Average Precision (AP) tóm tắt độ chính xác của
cảnh báo khi thay đổi ngưỡng. Tỷ lệ gian lận là mốc tham khảo cho AP ngẫu nhiên.
Thời gian fit được đo trong từng mô hình. Thời gian toàn lượt còn gồm tiền xử
lý và suy luận.

In [ ]:
MODEL_SLUG = 'ft-transformer'
output_dir = ROUND1_DIR / ('smoke' if SMOKE else 'full')
output_dir.mkdir(parents=True, exist_ok=True)
oof = np.full(len(y), np.nan, dtype=np.float64)
fold_metrics = []
run_start = perf_counter()
for fold, (train_idx, val_idx) in enumerate(splits[:1] if SMOKE else splits):
    print(f'Bắt đầu fold {fold + 1}/{1 if SMOKE else len(splits)}', flush=True)
    if np.isfinite(oof[val_idx]).any():
        raise RuntimeError('Fold xác thực bị trùng')
    scores, details = fit_predict(train_idx, val_idx, fold)
    scores = np.asarray(scores, dtype=np.float64)
    if (scores.shape != (len(val_idx),) or not np.isfinite(scores).all()
            or (scores < 0).any() or (scores > 1).any()):
        raise RuntimeError(f'Fold {fold + 1} trả về xác suất không hợp lệ')
    if 'RTX 4060' not in details['device']:
        raise RuntimeError(f"Fold {fold + 1} không xác minh được RTX 4060: {details['device']}")
    oof[val_idx] = scores
    fold_auc = float(roc_auc_score(y[val_idx], scores))
    record = dict(details, fold=fold + 1, validation_rows=len(val_idx), roc_auc=fold_auc)
    fold_metrics.append(record)
    print(f"Fold {fold + 1}: ROC AUC {fold_auc:.6f} | {details['fit_seconds']:.2f}s | {details['device']}", flush=True)

evaluated = np.isfinite(oof)
if not SMOKE and not evaluated.all():
    raise RuntimeError('Dự đoán OOF chưa phủ toàn bộ tập huấn luyện')
metrics = {
    'model': MODEL_SLUG,
    'status': 'completed',
    'smoke': SMOKE,
    'models_use_llm': False,
    'rows': int(evaluated.sum()),
    'positives': int(y[evaluated].sum()),
    'train_sha256': DATA_HASH,
    'split_sha256': SPLIT_HASH,
    'roc_auc': float(roc_auc_score(y[evaluated], oof[evaluated])),
    'average_precision': float(average_precision_score(y[evaluated], oof[evaluated])),
    'fit_seconds': float(sum(record['fit_seconds'] for record in fold_metrics)),
    'wall_seconds': float(perf_counter() - run_start),
    'folds': fold_metrics,
    'versions': {name: sys.modules[name].__version__
                 for name in ('numpy', 'pandas', 'sklearn', 'torch', 'lightgbm', 'xgboost')
                 if name in sys.modules},
}
result = output_dir / f'{MODEL_SLUG}.json'
temp = result.with_suffix('.tmp')
temp.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
temp.replace(result)
print(f"{'Smoke AUC' if SMOKE else 'ROC AUC OOF'}: {metrics['roc_auc']:.6f}")
print(f"AP: {metrics['average_precision']:.6f} | Fit: {metrics['fit_seconds']:.2f}s")